In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
import numpy as np

R = 8.314462618
T = 298.15
F = 96485.33212
n = 2
E0 = 1.49

data_path = Path.cwd() / "data" / "HOCl_Nernst_pH_Example.csv"
df = pd.read_csv(data_path)

# Convert mg/L to mol/L
HOCl = (df["HOCl_mg_L"] / 1000) / 52.46
Cl = (df["chloride_mg_L"] / 1000) / 35.45
H = 10 ** (-df["pH"])

# HOCl + H+ + 2e- -> Cl- + H2O
Q = Cl / (HOCl * H)

df["nernst_voltage_V"] = E0 - (R * T / (n * F)) * np.log(Q)
df["nernst_signal_mV"] = df["nernst_voltage_V"] * 1000

df[["pH", "HOCl_mg_L", "nernst_signal_mV"]].head()

import plotly.graph_objects as go

import statsmodels.api as sm
from pathlib import Path
import pandas as pd

data_path = Path.cwd() / "data" / "FreeChlorine_Nernst_Example.csv"
df = pd.read_csv(data_path)
df.head()
x = df["free_chlorine_mg_L"]
y = df["signal_raw_mV"]

df.head()

,sensor_num,pH,free_chlorine_mg_L,chloride_mg_L,signal_raw_mV
0,1,7.0,0.2,20.0,724.7
1,1,7.0,0.5,20.0,736.4
2,1,7.0,1.0,20.0,745.6
3,1,7.0,2.0,20.0,754.9
4,1,7.0,5.0,20.0,766.8


## Access CSV file

In [19]:
# Check Path and File
data_dir = Path.cwd() / "data"
print (data_dir)
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory does not exist: {data_dir}")
csv_files = list(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {data_dir}")
print("CSV files found:")
for f in csv_files:
    print(" -", f.name)

# Complete the data path
data_dir = Path.cwd() / "data" / "ChlorineSensor_free_chlorine_mg_LCalibrationCurve_Measures.csv"

c:\Users\SilminaWota\Documents\chlorinesensor\data
CSV files found:
 - ChlorineSensor_pHCalibrationCurve_Measures.csv
 - ChlorineSensor_TemperatureCalibrationCurve_Measures.csv
 - FreeChlorine_Nernst_Example.csv
 - HOCl_Nernst_pH_Example.csv


In [20]:
# Load the data in a dateframe
df = pd.read_csv(data_dir)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\SilminaWota\\Documents\\chlorinesensor\\data\\ChlorineSensor_free_chlorine_mg_LCalibrationCurve_Measures.csv'

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   sensor_num  14 non-null     int64  
 1   pH          14 non-null     float64
 2   signal_raw  14 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 468.0 bytes


## Signal analysis

In [ ]:
df_electrode1 = df[df["sensor_num"] == 1]
df_electrode2 = df[df["sensor_num"] == 2]

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_electrode1["free_chlorine_mg_L"],
        y=df_electrode1["signal_raw"],
        mode="markers",
        name="Chlorine sensor 1",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_electrode2["free_chlorine_mg_L"],
        y=df_electrode2["signal_raw"],
        mode="markers",
        name="Chlorine sensor 2",
    )
)

fig.update_layout(
    title="Calibration curves: free_chlorine_mg_L vs signal",
    xaxis_title="free_chlorine_mg_L",
    yaxis_title="Signal",
)

fig.show()

In [ ]:
# Sensor 1
coef_1 = np.polyfit(df_electrode1["free_chlorine_mg_L"], df_electrode1["signal_raw"], deg=1)
fit_1 = np.poly1d(coef_1)
a1, b1 = coef_1   # sensor 1

# Sensor 2
coef_2 = np.polyfit(df_electrode2["free_chlorine_mg_L"], df_electrode2["signal_raw"], deg=1)
fit_2 = np.poly1d(coef_2)
a2, b2 = coef_2   # sensor 2

print(f"Sensor 1 slope: {a1:.3f}, intercept:{b1:.3f}")
print(f"Sensor 2 slope: {a2:.3f}, intercept:{b2:.3f}")

Sensor 1 slope: -18.918, intercept:393.325
Sensor 2 slope: -19.307, intercept:411.107


In [ ]:
temp_range = np.linspace(
    df["free_chlorine_mg_L"].min(),
    df["free_chlorine_mg_L"].max(),
    100
)

fig = go.Figure()

# Raw points
fig.add_trace(go.Scatter(
    x=df_electrode1["free_chlorine_mg_L"], y=df_electrode1["signal_raw"],
    mode="markers", name="Chlorine sensor 1"
))
fig.add_trace(go.Scatter(
    x=df_electrode2["free_chlorine_mg_L"], y=df_electrode2["signal_raw"],
    mode="markers", name="Chlorine sensor 2"
))

# Fit lines
fig.add_trace(go.Scatter(
    x=temp_range, y=fit_1(temp_range),
    mode="lines", name="Calibration Sensor 1"
))
fig.add_trace(go.Scatter(
    x=temp_range, y=fit_2(temp_range),
    mode="lines", name="Calibration Sensor 2"
))

fig.update_layout(
    title="Calibration curves (linear fit)",
    xaxis_title="free_chlorine_mg_L",
    yaxis_title="Signal",
)

fig.show()


In [ ]:
# Statistical metrics
delta_slope = a2 - a1 # for sensitivity difference
#delta_intercept = b2 - b1 # for offset / bias
print(f"Sensitivity difference = {delta_slope:.3f}, difference in slope")
#print(f"Offset = {delta_intercept:.3f}")

def r_squared(x, y, model):
    y_pred = model(x)
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot

r2_1 = r_squared(df_electrode1["free_chlorine_mg_L"], df_electrode1["signal_raw"], fit_1)
r2_2 = r_squared(df_electrode2["free_chlorine_mg_L"], df_electrode2["signal_raw"], fit_2)

print(f"Sensor 1: R² = {r2_1:.3f}")
print(f"Sensor 2: R² = {r2_2:.3f}")

Sensitivity difference = -0.389, difference in slope
Sensor 1: R² = 1.000
Sensor 2: R² = 1.000


# Global calibration

In [ ]:
x_all = df["free_chlorine_mg_L"]
y_all = df["signal_raw"]

coef_global = np.polyfit(x_all, y_all, deg=1)
fit_global = np.poly1d(coef_global)

a_g, b_g = coef_global

df["residual_global"] = df["signal_raw"] - fit_global(df["free_chlorine_mg_L"])
df.groupby("sensor_num")["residual_global"].agg(["mean", "std"])

df_coef = pd.DataFrame({
    "sensor": ["Sensor 1", "Sensor 2", "Global"],
    "slope": [a1, a2, a_g],
    "intercept": [b1, b2, b_g],
    "R2": [r2_1, r2_2, r_squared(x_all, y_all, fit_global)],
})

df_coef.head()

,sensor,slope,intercept,R2
0,Sensor 1,-18.917857,393.325000,0.999871
1,Sensor 2,-19.307143,411.107143,0.999876
2,Global,-19.112500,402.216071,0.962444


In [ ]:
# plot with confidence interval (CI) - uncertainty around the mean fitted line
# x/y as arrays
x = df["free_chlorine_mg_L"].to_numpy()
y = df["signal_raw"].to_numpy()

# Fit OLS: y = b + a*x
X = sm.add_constant(x)          # adds intercept column
model = sm.OLS(y, X).fit()

# free_chlorine_mg_L grid for the line
x_grid = np.linspace(x.min(), x.max(), 200)
Xg = sm.add_constant(x_grid)

# Predictions + 95% CI
pred = model.get_prediction(Xg).summary_frame(alfree_chlorine_mg_La=0.05)
y_mean  = pred["mean"].to_numpy()
y_low   = pred["mean_ci_lower"].to_numpy()
y_high  = pred["mean_ci_upper"].to_numpy()

# ---- Plot ----
fig = go.Figure()

# Data points (colored by sensor_num)
for s in sorted(df["sensor_num"].dropna().unique()):
    sub = df[df["sensor_num"] == s]
    fig.add_trace(go.Scatter(
        x=sub["free_chlorine_mg_L"], y=sub["signal_raw"],
        mode="markers",
        name=f"Chlorine sensor {int(s)}"
    ))

# CI band (fill between low/high)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_grid, x_grid[::-1]]),
    y=np.concatenate([y_high, y_low[::-1]]),
    fill="toself",
    name="95% CI (mean)",
    line=dict(width=0),
    opacity=0.2,
))

# Global fit line
fig.add_trace(go.Scatter(
    x=x_grid, y=y_mean,
    mode="lines",
    name="Global calibration",
))

# Show coefficients in title (3 decimals)
b, a = model.params  # const, slope
fig.update_layout(
    title=f"Global calibration: signal = {a:.3f}·T + {b:.3f}",
    xaxis_title="free_chlorine_mg_L",
    yaxis_title="Signal"
)

fig.show()
fig.write_image("plot_name.png", width=1200, height=600)

fig.write_image("plot_name.png", width=1200, height=600)